## 1. Initial Setup

In [ ]:
from pathlib import Path
import os, json, re
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

ROOT = Path(".")
POLICY_DIR = ROOT / "data" / "healthcare_policies"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

load_dotenv(".env", override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model = os.getenv("AZURE_OPENAI_MODEL")
embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

if not all([endpoint, api_key, model, embedding_model]):
    raise ValueError(
        "Missing Azure configuration. Check AZURE_OPENAI_ENDPOINT, "
        "AZURE_OPENAI_API_KEY, AZURE_OPENAI_MODEL and "
        "AZURE_OPENAI_EMBEDDING_MODEL in .env"
    )

embeddings = OpenAIEmbeddings(
    model=embedding_model,
    base_url=endpoint,
    api_key=api_key,
)

llm = ChatOpenAI(
    model=model,
    base_url=endpoint,
    api_key=api_key,
    temperature=0,
)

chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=82,
    add_start_index=True,
)

manifest = json.loads((POLICY_DIR / "policy_manifest.json").read_text(encoding="utf-8"))
manifest_df = pd.DataFrame(manifest)

env_df = pd.DataFrame([
    {"Component": "Chat / Generation", "LangChain class": "ChatOpenAI", "Deployment": model, "Purpose": "Reranking and retrieval analysis"},
    {"Component": "Embeddings", "LangChain class": "OpenAIEmbeddings", "Deployment": embedding_model, "Purpose": "Semantic chunking and vector retrieval"},
    {"Component": "Chunking", "LangChain class": "SemanticChunker", "Deployment": embedding_model, "Purpose": "Embedding-driven chunk boundaries"},
])

display(env_df)
display(
    manifest_df[
        ["doc_id", "title", "plan_type", "policy_domain", "effective_date", "filename"]
    ].rename(columns={
        "doc_id": "Document ID",
        "title": "Policy",
        "plan_type": "Plan",
        "policy_domain": "Domain",
        "effective_date": "Effective Date",
        "filename": "Source File"
    })
)


In [ ]:
SECTION_PATTERN = re.compile(r"SECTION\s+\d+:\s+[A-Z0-9 &/\-]+", re.I)

def clean_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def infer_section_heading(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in lines[:5]:
        if SECTION_PATTERN.fullmatch(line):
            return line.upper()
        if len(line) <= 80 and line == line.upper() and any(ch.isalpha() for ch in line):
            return line
    match = SECTION_PATTERN.search(text)
    return match.group(0).upper() if match else "BODY TEXT"

docs = {}
page_rows = []

for item in manifest:
    loader = PyPDFLoader(str(POLICY_DIR / item["filename"]))
    raw_pages = loader.load()

    page_docs = []
    for page_doc in raw_pages:
        page_no = int(page_doc.metadata.get("page", 0)) + 1
        cleaned = clean_text(page_doc.page_content)
        normalized = Document(
            page_content=cleaned,
            metadata={
                "doc_id": item["doc_id"],
                "title": item["title"],
                "plan_type": item["plan_type"],
                "policy_domain": item["policy_domain"],
                "effective_date": item["effective_date"],
                "source_file": item["filename"],
                "page": page_no,
            },
        )
        page_docs.append(normalized)
        page_rows.append({
            "doc_id": item["doc_id"],
            "plan_type": item["plan_type"],
            "page": page_no,
            "characters": len(cleaned),
            "words": len(cleaned.split()),
            "preview": re.sub(r"\s+", " ", cleaned)[:120] + ("..." if cleaned else "")
        })

    docs[item["doc_id"]] = {"metadata": item, "pages": page_docs}

page_df = pd.DataFrame(page_rows)
display(page_df)


In [ ]:
sample_doc_id = "GOLD-PPO-2026"
sample_filename = manifest_df.loc[manifest_df["doc_id"] == sample_doc_id, "filename"].iloc[0]
raw_loader = PyPDFLoader(str(POLICY_DIR / sample_filename))
raw_pages = raw_loader.load()
raw_sample = "\n".join(page.page_content for page in raw_pages)
clean_sample = "\n".join(page.page_content for page in docs[sample_doc_id]["pages"])

comparison_df = pd.DataFrame([
    {
        "Version": "Raw extracted text",
        "Characters": len(raw_sample),
        "Preview": re.sub(r"\s+", " ", raw_sample)[:180] + "..."
    },
    {
        "Version": "Minimally cleaned text",
        "Characters": len(clean_sample),
        "Preview": re.sub(r"\s+", " ", clean_sample)[:180] + "..."
    }
])

display(comparison_df)


Chunking

In [ ]:
chunker_config_df = pd.DataFrame([
    {"Setting": "Chunker", "Value": "SemanticChunker"},
    {"Setting": "Embedding model", "Value": embedding_model},
    {"Setting": "Breakpoint threshold type", "Value": "percentile"},
    {"Setting": "Breakpoint threshold amount", "Value": 82},
    {"Setting": "add_start_index", "Value": True},
    {"Setting": "Chunking scope", "Value": "Semantic chunking within each page document"},
])

display(chunker_config_df)

def semantic_chunk_page_documents(page_docs):
    chunk_docs = []
    for page_doc in page_docs:
        splits = chunker.split_documents([page_doc])
        for idx, split_doc in enumerate(splits):
            chunk_text = clean_text(split_doc.page_content)
            if not chunk_text:
                continue
            chunk_docs.append(
                Document(
                    page_content=chunk_text,
                    metadata={
                        **page_doc.metadata,
                        "chunk_strategy": "semantic",
                        "chunk_index_on_page": idx,
                        "section": infer_section_heading(chunk_text),
                    },
                )
            )
    return chunk_docs


In [ ]:
sample_pages = docs[sample_doc_id]["pages"]
sample_chunks = semantic_chunk_page_documents(sample_pages)

sample_summary_df = pd.DataFrame([
    {
        "Sample Document": sample_doc_id,
        "Pages": len(sample_pages),
        "Semantic Chunks": len(sample_chunks),
        "Avg Chunk Chars": round(np.mean([len(doc.page_content) for doc in sample_chunks]), 1),
        "Median Chunk Chars": round(np.median([len(doc.page_content) for doc in sample_chunks]), 1),
    }
])

sample_preview_rows = []
for i, chunk_doc in enumerate(sample_chunks[:8], start=1):
    sample_preview_rows.append({
        "Chunk #": i,
        "Page": chunk_doc.metadata["page"],
        "Section": chunk_doc.metadata["section"],
        "Characters": len(chunk_doc.page_content),
        "Words": len(chunk_doc.page_content.split()),
        "Starts With": re.sub(r"\s+", " ", chunk_doc.page_content)[:220] + ("..." if len(chunk_doc.page_content) > 220 else "")
    })

per_page_chunk_df = pd.DataFrame([
    {
        "Page": page_doc.metadata["page"],
        "Page Characters": len(page_doc.page_content),
        "Semantic Chunks On Page": sum(1 for chunk_doc in sample_chunks if chunk_doc.metadata["page"] == page_doc.metadata["page"])
    }
    for page_doc in sample_pages
])

display(sample_summary_df)
display(per_page_chunk_df)
display(pd.DataFrame(sample_preview_rows))


## 6. Build the full semantic chunk corpus with LangChain

In [ ]:
all_semantic_docs = []
semantic_records = []
corpus_summary_rows = []

for doc_id, doc_bundle in docs.items():
    page_docs = doc_bundle["pages"]
    chunk_docs = semantic_chunk_page_documents(page_docs)
    all_semantic_docs.extend(chunk_docs)

    corpus_summary_rows.append({
        "Document": doc_id,
        "Plan": doc_bundle["metadata"]["plan_type"],
        "Pages": len(page_docs),
        "Semantic Chunks": len(chunk_docs),
        "Avg Chunk Chars": round(np.mean([len(chunk.page_content) for chunk in chunk_docs]), 1),
    })

for chunk_doc in all_semantic_docs:
    chunk_id = f"{chunk_doc.metadata['doc_id']}-{int(chunk_doc.metadata['page']):02d}-semantic-{int(chunk_doc.metadata['chunk_index_on_page']):03d}"
    chunk_doc.metadata["chunk_id"] = chunk_id

    semantic_records.append({
        "chunk_id": chunk_id,
        "doc_id": chunk_doc.metadata["doc_id"],
        "source_file": chunk_doc.metadata["source_file"],
        "title": chunk_doc.metadata["title"],
        "plan_type": chunk_doc.metadata["plan_type"],
        "policy_domain": chunk_doc.metadata["policy_domain"],
        "effective_date": chunk_doc.metadata["effective_date"],
        "page": int(chunk_doc.metadata["page"]),
        "section": chunk_doc.metadata["section"],
        "chunk_strategy": "semantic",
        "char_count": len(chunk_doc.page_content),
        "word_count": len(chunk_doc.page_content.split()),
        "text": chunk_doc.page_content,
    })

display(pd.DataFrame(corpus_summary_rows))

relationship_df = pd.DataFrame(semantic_records)[
    ["chunk_id", "doc_id", "plan_type", "section", "page", "char_count", "source_file", "text"]
].copy()
relationship_df["text"] = relationship_df["text"].str.replace(r"\s+", " ", regex=True).str[:180] + "..."
display(relationship_df.head(12))


## 7. Inspect one business topic inside the semantic chunk corpus

In [ ]:
needle = "physical therapy"

topic_df = pd.DataFrame(semantic_records)
topic_df = topic_df[topic_df["text"].str.lower().str.contains(needle, na=False)].copy()
topic_df["Policy Text"] = topic_df["text"].str.replace(r"\s+", " ", regex=True).str[:260] + "..."

display(
    topic_df[
        ["chunk_id", "doc_id", "plan_type", "section", "page", "char_count", "Policy Text"]
    ].head(8)
)


> **Observation:** The policy text did not change. The LangChain semantic chunker changed the unit that retrieval will later search.

## 8. Persist the semantic corpus for the retrieval steps

In [ ]:
def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in records:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

semantic_artifact = ARTIFACT_DIR / "policy_chunks_semantic_langchain.jsonl"
save_jsonl(semantic_records, semantic_artifact)

display(pd.DataFrame([
    {
        "Strategy": "semantic",
        "Framework": "LangChain",
        "Artifact": str(semantic_artifact),
        "Chunks": len(semantic_records),
        "Avg Chars": round(np.mean([record["char_count"] for record in semantic_records]), 1),
    }
]))


## 9. What does a LangChain embedding look like?

In [ ]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sample_texts = [
    "MRI scans require prior authorization.",
    "Advanced imaging needs insurer approval.",
    "The member updated a mailing address.",
]

sample_vectors = embeddings.embed_documents(sample_texts)

embedding_df = pd.DataFrame([
    {
        "Text": text,
        "Vector Dimensions": len(vec),
        "First 6 Values": str([round(v, 4) for v in vec[:6]]),
        "Vector Norm": round(float(np.linalg.norm(vec)), 4),
    }
    for text, vec in zip(sample_texts, sample_vectors)
])

pairs = [
    (0, 1, "Same business meaning, different wording"),
    (0, 2, "Different business meaning"),
    (1, 2, "Different business meaning"),
]

similarity_rows = []
for i, j, relationship in pairs:
    similarity_rows.append({
        "Text A": sample_texts[i],
        "Text B": sample_texts[j],
        "Expected Relationship": relationship,
        "Cosine Similarity": round(cosine_similarity(sample_vectors[i], sample_vectors[j]), 4),
    })

display(embedding_df)
display(pd.DataFrame(similarity_rows))


An embedding is a numeric representation of meaning. We do not usually interpret dimensions directly; we compare vectors geometrically.

For the Chroma setup below:
- lower distance is better
- higher similarity is better


## 10. Build the LangChain Chroma vector store

In [ ]:
VECTOR_DB_PATH = str(ARTIFACT_DIR / "chroma_policy_db_semantic_langchain")
COLLECTION_NAME = "policy_semantic_langchain"

vectorstore = Chroma.from_documents(
    documents=all_semantic_docs,
    embedding=embeddings,
    ids=[doc.metadata["chunk_id"] for doc in all_semantic_docs],
    persist_directory=VECTOR_DB_PATH,
    collection_name=COLLECTION_NAME,
    collection_metadata={"hnsw:space": "cosine"},
)

display(pd.DataFrame([
    {
        "Collection": COLLECTION_NAME,
        "Framework": "LangChain Chroma",
        "Vectors Stored": len(all_semantic_docs),
        "Vector DB Path": VECTOR_DB_PATH,
        "Avg Source Characters": round(np.mean([len(doc.page_content) for doc in all_semantic_docs]), 1),
    }
]))


## 11. Ranked semantic retrieval with LangChain

In [ ]:
def semantic_search(question, top_k=5, metadata_filter=None):
    return vectorstore.similarity_search_with_score(
        query=question,
        k=top_k,
        filter=metadata_filter,
    )

def search_to_df(results):
    rows = []
    for rank, (doc, score) in enumerate(results, start=1):
        rows.append({
            "Rank": rank,
            "Cosine Distance Down": round(float(score), 4),
            "Cosine Similarity Up": round(1 - float(score), 4),
            "Document": doc.metadata["doc_id"],
            "Plan": doc.metadata["plan_type"],
            "Section": doc.metadata["section"],
            "Page": doc.metadata["page"],
            "Chunk Strategy": doc.metadata["chunk_strategy"],
            "Retrieved Text": re.sub(r"\s+", " ", doc.page_content)[:240] + ("..." if len(doc.page_content) > 240 else ""),
        })
    return pd.DataFrame(rows)

question = "For Gold PPO, when does physical therapy start requiring authorization?"
result_df = search_to_df(semantic_search(question, top_k=5))
display(result_df)


### Reading this ranking

- **Rank 1** is the nearest semantic result returned by the LangChain vector store.
- **Similarity** is useful for within-system comparison, not as a universal confidence score.
- High semantic similarity does not automatically mean the text is the right answer.


## 12. Metadata filtering - show before and after

In [ ]:
query = "When does physical therapy require authorization?"

before = search_to_df(semantic_search(query, top_k=5))
after = search_to_df(
    semantic_search(
        query,
        top_k=5,
        metadata_filter={"plan_type": "Silver HMO"},
    )
)

before["Search Mode"] = "Semantic only"
after["Search Mode"] = "Semantic + Silver HMO filter"

display(
    pd.concat([before, after], ignore_index=True)[
        ["Search Mode", "Rank", "Cosine Similarity Up", "Document", "Plan", "Section", "Retrieved Text"]
    ]
)


> **Takeaway:** Metadata filtering changes the candidate universe before ranking. It helps semantic retrieval stay inside the correct plan or policy slice.

## 13. Semantic-only retrieval diagnostics across several questions

In [ ]:
questions = [
    "What information is needed for continued physical therapy authorization?",
    "How many chiropractic visits does Gold PPO allow each year?",
    "When is prior authorization required for MRI imaging?",
]

diagnostic_rows = []
for question in questions:
    top_df = search_to_df(semantic_search(question, top_k=3))
    top_row = top_df.iloc[0]
    diagnostic_rows.append({
        "Question": question,
        "Top Document": top_row["Document"],
        "Top Plan": top_row["Plan"],
        "Top Section": top_row["Section"],
        "Top Page": top_row["Page"],
        "Top Similarity": top_row["Cosine Similarity Up"],
        "Top Text": top_row["Retrieved Text"],
    })

display(pd.DataFrame(diagnostic_rows))


This replaces the original chunking-strategy comparison section. The chunking method stays fixed; only the question changes.

> **Teaching question:** When the query shifts from utilization management to benefit limits, does the top semantic chunk move to the right policy section?

## 14. Ranking vs reranking with a LangChain chain

In [ ]:
rerank_prompt = ChatPromptTemplate.from_template(
    """
You are reranking retrieved healthcare policy chunks for a user question.

QUESTION:
{question}

CANDIDATES:
{candidates_json}

Score every candidate from 0 to 100 for how directly it contains evidence needed to answer the question.
Return only a JSON array in this format:
[
  {{"chunk_id":"C1","relevance_score":95,"reason":"..."}}
]
Do not omit any candidate.
"""
)

rerank_chain = rerank_prompt | llm | JsonOutputParser()

def rerank_with_langchain(question, initial_df):
    candidates = []
    for _, row in initial_df.iterrows():
        candidates.append({
            "rank": int(row["Rank"]),
            "chunk_id": f"C{int(row['Rank'])}",
            "document": row["Document"],
            "section": row["Section"],
            "page": int(row["Page"]),
            "text": row["Retrieved Text"],
        })

    scores = rerank_chain.invoke(
        {
            "question": question,
            "candidates_json": json.dumps(candidates, indent=2),
        }
    )
    score_map = {item["chunk_id"]: item for item in scores}

    reranked = initial_df.copy()
    reranked["Candidate"] = [f"C{x}" for x in reranked["Rank"]]
    reranked["Rerank Score Up"] = reranked["Candidate"].map(
        lambda value: score_map.get(value, {}).get("relevance_score", 0)
    )
    reranked["Rerank Reason"] = reranked["Candidate"].map(
        lambda value: score_map.get(value, {}).get("reason", "")
    )

    reranked = reranked.sort_values(
        ["Rerank Score Up", "Cosine Similarity Up"], ascending=[False, False]
    ).reset_index(drop=True)

    reranked["Rank After Rerank"] = np.arange(1, len(reranked) + 1)
    reranked["Rank Movement"] = reranked["Rank"] - reranked["Rank After Rerank"]
    return reranked

rerank_question = "What clinical documentation is required when requesting continued physical therapy?"
initial = search_to_df(semantic_search(rerank_question, top_k=6))

display(
    initial[
        ["Rank", "Cosine Similarity Up", "Document", "Section", "Page", "Retrieved Text"]
    ].rename(columns={"Rank": "Vector Rank"})
)


In [ ]:
reranked = rerank_with_langchain(rerank_question, initial)

display(
    reranked[
        ["Rank", "Rank After Rerank", "Rank Movement", "Cosine Similarity Up",
         "Rerank Score Up", "Document", "Section", "Page", "Rerank Reason"]
    ].rename(columns={"Rank": "Vector Rank"})
)


### The reranking output

- **Vector Rank** = broad semantic closeness from the vector store.
- **Rerank Score** = deeper question-to-chunk relevance from the LangChain LLM chain.
- **Rank Movement > 0** = chunk moved upward after reranking.
- **Rank Movement < 0** = chunk moved downward.

> **Takeaway:** Retrieval finds candidates; reranking decides which candidates deserve the top context positions.

## 15. Top-k - convert the trade-off into a visible summary

In [ ]:
rows = []
query = "Gold PPO chiropractic annual visit limit"

for k in [1, 3, 5, 8]:
    df = search_to_df(semantic_search(query, top_k=k))
    rows.append({
        "Top-K": k,
        "Best Similarity": df["Cosine Similarity Up"].max(),
        "Unique Documents": df["Document"].nunique(),
        "Unique Plans": df["Plan"].nunique(),
        "Retrieved Characters": int(df["Retrieved Text"].str.len().sum()),
        "Top Result": f"{df.iloc[0]['Document']} | {df.iloc[0]['Section']}",
    })

display(pd.DataFrame(rows))


> **Takeaway:** Top-k is a recall-versus-noise control. Higher k expands the evidence pool, but it also increases context size, token cost and distraction.

## Day 1 completion

You can now explain the full retrieval workflow visually with LangChain:

**PyPDFLoader -> page documents -> SemanticChunker -> metadata-rich semantic chunks -> OpenAIEmbeddings -> Chroma -> similarity ranking -> metadata filtering -> ChatOpenAI reranking -> top-k**

This notebook is intentionally single-method and single-framework: every step uses LangChain and every retrieval result comes from semantic chunking.